# NUFROST Result Summary & Paper Figures (Local)

This notebook reads the local CSV results from the unified evaluation workflow and generates summary figures and tables for analysis and paper drafting.

**Expected CSV files:**
1. `data/output/hls_ablation_results.csv` – ablation study (NUFROST variants + baselines)
2. `data/output/hls_gap_sweep_results.csv` – continuous-gap sweep results
3. `data/output/hls_sparse_sweep_results.csv` – sparse-observation sweep results
4. `data/output/hls_repeatability_results.csv` – raw repeatability data
5. `data/output/repeatability_summary.csv` – optional derived repeatability summary

All figures are saved under `data/output/figures/`.


## 1. Local Configuration


In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "data/output"
FIGURE_DIR = OUTPUT_DIR / "figures"

CSV_ABLATION = OUTPUT_DIR / "hls_ablation_results.csv"
CSV_GAP_SWEEP = OUTPUT_DIR / "hls_gap_sweep_results.csv"
CSV_SPARSE_SWEEP = OUTPUT_DIR / "hls_sparse_sweep_results.csv"
CSV_REPEAT_RAW = OUTPUT_DIR / "hls_repeatability_results.csv"
CSV_REPEAT_SUMMARY = OUTPUT_DIR / "repeatability_summary.csv"

print(f"[Project dir: {PROJECT_DIR}]")
print(f"[Output dir: {OUTPUT_DIR}]")


In [ ]:
import os

os.chdir(PROJECT_DIR)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print(f"[Working directory changed to: {os.getcwd()}]")


## 2. Install Dependencies

In [ ]:
print("Install plotting dependencies in your local environment before running this notebook.")
print("Recommended: conda env create -f environment.yml or pip install -r requirements.txt seaborn")


## 3. Import Modules

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
from pathlib import Path
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

# Set plotting style
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

# Ensure figure directory exists
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Figure directory: {FIGURE_DIR}")


## 4. Load Data

In [ ]:
def load_csv_or_warn(path, name):
    if path.exists():
        df = pd.read_csv(path)
        print(f"✓ {name}: {len(df)} rows, columns: {list(df.columns)}")
        return df
    else:
        print(f"✗ {name}: File not found at {path}")
        return None

print("Loading CSV files...")
df_ablation = load_csv_or_warn(CSV_ABLATION, "Ablation results")
df_gap = load_csv_or_warn(CSV_GAP_SWEEP, "Gap‑length sweep")
df_sparse = load_csv_or_warn(CSV_SPARSE_SWEEP, "Sparse‑observation sweep")
df_repeat_raw = load_csv_or_warn(CSV_REPEAT_RAW, "Repeatability raw")
df_repeat_summary = load_csv_or_warn(CSV_REPEAT_SUMMARY, "Repeatability summary")

# If repeatability summary doesn't exist, compute it from raw data
if df_repeat_raw is not None and df_repeat_summary is None:
    print("\nComputing repeatability summary from raw data...")
    grouped = df_repeat_raw.groupby(["Image", "Scenario", "Algorithm"])[["RMSE", "MAE", "R", "OutlierRatio"]]
    df_mean = grouped.mean().round(4)
    df_std = grouped.std().round(4)
    summary = pd.DataFrame()
    for metric in ["RMSE", "MAE", "R", "OutlierRatio"]:
        summary[f"{metric}_mean"] = df_mean[metric]
        summary[f"{metric}_std"] = df_std[metric]
    df_repeat_summary = summary.reset_index()
    df_repeat_summary.to_csv(CSV_REPEAT_SUMMARY, index=False)
    print(f"✓ Repeatability summary saved to {CSV_REPEAT_SUMMARY}")


## 5. Ablation Study: Bar Plots

In [ ]:
if df_ablation is not None:
    print("\n=== Ablation Study ===")
    
    # Separate NuFrost variants from baseline methods
    nufrost_variants = df_ablation[df_ablation["Algorithm"] == "NuFrost"]["Variant"].unique()
    baseline_methods = ["Zhu2015", "HANTS"]
    
    # For each scenario (random/gap), create bar plots
    for scenario in ["random", "gap"]:
        df_scenario = df_ablation[df_ablation["Scenario"] == scenario]
        if df_scenario.empty:
            print(f"No data for scenario '{scenario}'", end=" ")
            continue
        
        # Compute mean metrics across all images
        df_mean = df_scenario.groupby(["Variant", "Algorithm"])[["RMSE", "MAE", "R", "OutlierRatio"]].mean().reset_index()
        
        # Sort variants: Full NUFROST first, then ablation variants, then baselines
        variant_order = ["Full NUFROST"] + [v for v in nufrost_variants if v != "Full NUFROST"] + baseline_methods
        df_mean["Variant"] = pd.Categorical(df_mean["Variant"], categories=variant_order, ordered=True)
        df_mean = df_mean.sort_values("Variant")
        
        # Create bar plots for each metric
        metrics = ["RMSE", "MAE", "R", "OutlierRatio"]
        n_metrics = len(metrics)
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        axes = axes.ravel()
        
        for idx, metric in enumerate(metrics):
            ax = axes[idx]
            # Color bars differently: Full NUFROST in green, ablation variants in orange, baselines in blue
            colors = []
            for var in df_mean["Variant"]:
                if var == "Full NUFROST":
                    colors.append("#2ecc71")  # green
                elif var in baseline_methods:
                    colors.append("#3498db")  # blue
                else:
                    colors.append("#f39c12")  # orange
            
            bars = ax.bar(df_mean["Variant"], df_mean[metric], color=colors, edgecolor="black", alpha=0.8)
            ax.set_title(f"{metric} ({scenario})", fontweight="bold", fontsize=12)
            ax.set_ylabel(metric, fontsize=11)
            ax.tick_params(axis="x", rotation=45, labelsize=10)
            ax.grid(axis="y", linestyle="--", alpha=0.3)
            
            # Add value labels on top of bars
            for bar in bars:
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height + 0.01*height,
                        f'{height:.3f}', ha='center', va='bottom', fontsize=9)
        
        plt.suptitle(f"NUFROST Ablation Study: {scenario} scenario", fontsize=14, fontweight="bold", y=1.02)
        plt.tight_layout()
        
        # Save figure
        fig_path = FIGURE_DIR / f"ablation_{scenario}.png"
        plt.savefig(fig_path, dpi=300, bbox_inches="tight")
        print(f"Saved: {fig_path}")
        plt.show()
        
        # Display summary table
        display(Markdown(f"**Summary table for {scenario} scenario**"))
        display(df_mean[['Variant', 'RMSE', 'MAE', 'R', 'OutlierRatio']].round(4))
else:
    print("Skipping ablation study (no data)")


## 6. Gap‑Length Sweep: Performance Curves

In [ ]:
if df_gap is not None:
    print("\n=== Gap‑Length Sweep ===")
    
    # Compute mean metrics per algorithm and gap length (across all images)
    df_mean = df_gap.groupby(["Algorithm", "GapLength"])[["RMSE", "MAE", "R", "OutlierRatio"]].mean().reset_index()
    
    # Plot each metric
    metrics = ["RMSE", "MAE", "R", "OutlierRatio"]
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.ravel()
    
    # Color palette for algorithms
    algorithm_colors = {"NuFrost": "#e74c3c", "Zhu2015": "#3498db", "HANTS": "#f39c12"}
    
    for idx, metric in enumerate(metrics):
        ax = axes[idx]
        for algo, color in algorithm_colors.items():
            df_algo = df_mean[df_mean["Algorithm"] == algo]
            if not df_algo.empty:
                # Sort by gap length
                df_algo = df_algo.sort_values("GapLength")
                ax.plot(df_algo["GapLength"], df_algo[metric], 
                        marker="o", markersize=6, linewidth=2.5, label=algo, color=color)
        
        ax.set_xlabel("Gap Length (days)", fontsize=12)
        ax.set_ylabel(metric, fontsize=12)
        ax.set_title(f"{metric} vs. Gap Length", fontweight="bold", fontsize=13)
        ax.legend(loc="best", fontsize=10)
        ax.grid(True, linestyle="--", alpha=0.3)
    
    plt.suptitle("Algorithm Performance vs. Continuous Gap Length", fontsize=15, fontweight="bold", y=1.02)
    plt.tight_layout()
    
    # Save figure
    fig_path = FIGURE_DIR / "gap_length_sweep.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    print(f"Saved: {fig_path}")
    plt.show()
    
    # Display a compact summary table (mean across gap lengths)
    display(Markdown("**Average performance across all gap lengths**"))
    summary_gap = df_gap.groupby("Algorithm")[["RMSE", "MAE", "R", "OutlierRatio"]].mean().round(4)
    display(summary_gap)
    
    # Also show performance at specific gap lengths (e.g., 60, 120, 180 days)
    for gap in [60, 120, 180, 240]:
        if gap in df_mean["GapLength"].unique():
            df_gap_specific = df_mean[df_mean["GapLength"] == gap]
            if not df_gap_specific.empty:
                display(Markdown(f"**Performance at {gap}-day gap**"))
                display(df_gap_specific[['Algorithm', 'RMSE', 'MAE', 'R', 'OutlierRatio']].round(4))
else:
    print("Skipping gap‑length sweep (no data)")


## 7. Sparse‑Observation Sweep: Performance Curves

In [ ]:
if df_sparse is not None:
    print("\n=== Sparse‑Observation Sweep ===")
    
    # Compute mean metrics per algorithm and number of masked points
    df_mean = df_sparse.groupby(["Algorithm", "NumPoints"])[["RMSE", "MAE", "R", "OutlierRatio"]].mean().reset_index()
    
    # Plot each metric
    metrics = ["RMSE", "MAE", "R", "OutlierRatio"]
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.ravel()
    
    algorithm_colors = {"NuFrost": "#e74c3c", "Zhu2015": "#3498db", "HANTS": "#f39c12"}
    
    for idx, metric in enumerate(metrics):
        ax = axes[idx]
        for algo, color in algorithm_colors.items():
            df_algo = df_mean[df_mean["Algorithm"] == algo]
            if not df_algo.empty:
                # Sort by NumPoints
                df_algo = df_algo.sort_values("NumPoints")
                ax.plot(df_algo["NumPoints"], df_algo[metric], 
                        marker="s", markersize=6, linewidth=2.5, label=algo, color=color)
        
        ax.set_xlabel("Number of Masked Points", fontsize=12)
        ax.set_ylabel(metric, fontsize=12)
        ax.set_title(f"{metric} vs. Sparsity", fontweight="bold", fontsize=13)
        ax.legend(loc="best", fontsize=10)
        ax.grid(True, linestyle="--", alpha=0.3)
        
        # Use logarithmic x‑axis for better visualization
        ax.set_xscale("log")
    
    plt.suptitle("Algorithm Performance vs. Random‑Point Masking", fontsize=15, fontweight="bold", y=1.02)
    plt.tight_layout()
    
    # Save figure
    fig_path = FIGURE_DIR / "sparse_observation_sweep.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    print(f"Saved: {fig_path}")
    plt.show()
    
    # Display summary table
    display(Markdown("**Average performance across all sparsity levels**"))
    summary_sparse = df_sparse.groupby("Algorithm")[["RMSE", "MAE", "R", "OutlierRatio"]].mean().round(4)
    display(summary_sparse)
    
    # Show performance at specific sparsity levels
    for points in [1000, 5000, 20000, 50000, 100000]:
        if points in df_mean["NumPoints"].unique():
            df_points_specific = df_mean[df_mean["NumPoints"] == points]
            if not df_points_specific.empty:
                display(Markdown(f"**Performance at {points} masked points**"))
                display(df_points_specific[['Algorithm', 'RMSE', 'MAE', 'R', 'OutlierRatio']].round(4))
else:
    print("Skipping sparse‑observation sweep (no data)")


## 8. Repeatability: Statistical Robustness

In [ ]:
if df_repeat_summary is not None:
    print("\n=== Repeatability Analysis ===")
    
    # Compute overall mean ± std across all images
    if df_repeat_raw is not None:
        overall_mean = df_repeat_raw.groupby(["Scenario", "Algorithm"])[["RMSE", "MAE", "R", "OutlierRatio"]].mean().round(4)
        overall_std = df_repeat_raw.groupby(["Scenario", "Algorithm"])[["RMSE", "MAE", "R", "OutlierRatio"]].std().round(4)
    else:
        # Fallback: compute from summary
        overall_mean = df_repeat_summary.groupby(["Scenario", "Algorithm"])[[col for col in df_repeat_summary.columns if col.endswith("_mean")]]
        overall_mean.columns = [col.replace("_mean", "") for col in overall_mean.columns]
        overall_std = df_repeat_summary.groupby(["Scenario", "Algorithm"])[[col for col in df_repeat_summary.columns if col.endswith("_std")]]
        overall_std.columns = [col.replace("_std", "") for col in overall_std.columns]
    
    # Create formatted tables for paper
    for scenario in ["random", "gap"]:
        display(Markdown(f"**Statistical robustness: {scenario} scenario**"))
        
        # Build a nice table with mean ± std
        table_data = []
        for algo in ["NuFrost", "Zhu2015", "HANTS"]:
            if (scenario, algo) in overall_mean.index:
                mu = overall_mean.loc[(scenario, algo)]
                sigma = overall_std.loc[(scenario, algo)]
                
                row = {"Algorithm": algo}
                for metric in ["RMSE", "MAE", "R", "OutlierRatio"]:
                    if metric in mu and metric in sigma:
                        row[metric] = f"{mu[metric]:.4f} ± {sigma[metric]:.4f}"
                table_data.append(row)
        
        if table_data:
            df_table = pd.DataFrame(table_data)
            display(df_table)
            
            # Save table to CSV
            table_path = FIGURE_DIR / f"repeatability_{scenario}_table.csv"
            df_table.to_csv(table_path, index=False)
            print(f"Saved table: {table_path}")
    
    # Create box plots for visual comparison
    if df_repeat_raw is not None:
        for scenario in ["random", "gap"]:
            df_scenario = df_repeat_raw[df_repeat_raw["Scenario"] == scenario]
            if df_scenario.empty:
                continue
            
            metrics = ["RMSE", "MAE", "R", "OutlierRatio"]
            fig, axes = plt.subplots(2, 2, figsize=(14, 10))
            axes = axes.ravel()
            
            for idx, metric in enumerate(metrics):
                ax = axes[idx]
                sns.boxplot(data=df_scenario, x="Algorithm", y=metric, ax=ax,
                           palette=["#e74c3c", "#3498db", "#f39c12"],
                           order=["NuFrost", "Zhu2015", "HANTS"])
                ax.set_title(f"{metric} ({scenario})", fontweight="bold", fontsize=12)
                ax.set_xlabel("")
                ax.set_ylabel(metric, fontsize=11)
                ax.grid(axis="y", linestyle="--", alpha=0.3)
                
                # Add mean markers
                for i, algo in enumerate(["NuFrost", "Zhu2015", "HANTS"]):
                    df_algo = df_scenario[df_scenario["Algorithm"] == algo]
                    if not df_algo.empty:
                        mean_val = df_algo[metric].mean()
                        ax.plot(i, mean_val, 'ko', markersize=6)
            
            plt.suptitle(f"Repeatability Analysis: {scenario} scenario (5 repeats)", fontsize=14, fontweight="bold", y=1.02)
            plt.tight_layout()
            
            fig_path = FIGURE_DIR / f"repeatability_boxplot_{scenario}.png"
            plt.savefig(fig_path, dpi=300, bbox_inches="tight")
            print(f"Saved: {fig_path}")
            plt.show()
else:
    print("Skipping repeatability analysis (no data)")


## 9. Export Summary Tables for Paper

In [ ]:
print("\n=== Exporting Summary Tables ===")

# Create a comprehensive summary table across all experiments
summary_tables = {}

# 1. Overall performance (averaged across all experiments where available)
if df_gap is not None:
    gap_summary = df_gap.groupby("Algorithm")[["RMSE", "MAE", "R", "OutlierRatio"]].mean().round(4)
    gap_summary.columns = [f"Gap_{col}" for col in gap_summary.columns]
    summary_tables["gap"] = gap_summary

if df_sparse is not None:
    sparse_summary = df_sparse.groupby("Algorithm")[["RMSE", "MAE", "R", "OutlierRatio"]].mean().round(4)
    sparse_summary.columns = [f"Sparse_{col}" for col in sparse_summary.columns]
    summary_tables["sparse"] = sparse_summary

if df_ablation is not None:
    # Get Full NUFROST performance
    nufrost_full = df_ablation[(df_ablation["Variant"] == "Full NUFROST") & (df_ablation["Scenario"] == "random")]
    if not nufrost_full.empty:
        ablation_summary = nufrost_full.groupby("Algorithm")[["RMSE", "MAE", "R", "OutlierRatio"]].mean().round(4)
        ablation_summary.columns = [f"Ablation_{col}" for col in ablation_summary.columns]
        summary_tables["ablation"] = ablation_summary

# Combine all summaries
if summary_tables:
    # Start with algorithm list
    algorithms = []
    for df in summary_tables.values():
        algorithms.extend(df.index.tolist())
    algorithms = sorted(set(algorithms))
    
    # Create combined dataframe
    combined = pd.DataFrame(index=algorithms)
    for name, df_table in summary_tables.items():
        combined = combined.join(df_table, how="outer")
    
    display(Markdown("**Combined Performance Summary**"))
    display(combined)
    
    # Save to CSV
    combined_path = FIGURE_DIR / "combined_summary.csv"
    combined.to_csv(combined_path)
    print(f"✓ Combined summary saved: {combined_path}")
else:
    print("No summary tables to combine.")


## 10. Summary

In [ ]:
print("\n========== Summary ==========")
print(f"Figures saved to: {FIGURE_DIR}")
print(f"Total CSV files loaded: {sum(1 for f in [df_ablation, df_gap, df_sparse, df_repeat_raw, df_repeat_summary] if f is not None)}")

# List generated files
if FIGURE_DIR.exists():
    print("\nGenerated files:")
    for file in sorted(FIGURE_DIR.glob("*")):
        if file.is_file():
            print(f"  • {file.name}")
else:
    print("No figures directory found.")
